# Problem 6: Moving Your Code to the RCC
---

Login to the RCC. You need to have 2-factor authentication enabled to sign in.

```
ssh <cnetid>@midway3.rcc.uchicago.edu
```

### Account Information
```
# Show you quotas and balance
accounts balance
```

### Bash file
Determine which shell you are using on `midway3`. It should be bash.
```
echo $SHELL
# /bin/bash
```

---
## Part 1: The login node

When you sign in you land on a **login node**. It is shared by everyone on Midway3 and is for editing files, setting up environments, and short tests. Longer work runs on **compute nodes**, which you reach through the Slurm scheduler; that is Module 4. The handbook's [The RCC cluster](https://uchicago-scientific.github.io/mpcs56430-handbook/rcc/) chapter covers every step of this problem.

Run `lscpu` on the login node. Report `Thread(s) per core`, `Socket(s)` and `Core(s) per socket` below:

Thread(s) per core:  2
Core(s) per socket:  16
Socket(s):           2

---

## Part 2: Move your code

Clone your own Module 3 repository on Midway3. GitHub needs to know this machine, so create an SSH key on Midway3 and add it to your GitHub account:

```
ssh-keygen -t ed25519 -C "<cnetid>@midway3"
cat ~/.ssh/id_ed25519.pub
```

Copy the line that `cat` prints into **GitHub → Settings → SSH and GPG keys → New SSH key**. Then clone:

```
git clone git@github.com:uchicago-scientific/<your-module-3-repository>.git
cd <your-module-3-repository>
```

Push your work from your laptop first, so the clone has your Problem 4 code.

## Part 3: Rebuild the environment

Install uv on Midway3, then rebuild the project from `pyproject.toml` and `uv.lock`, exactly as on your laptop:

```
curl -LsSf https://astral.sh/uv/install.sh | sh
source $HOME/.local/bin/env
uv sync --locked
uv run pytest
```

If anything fails here that works on your laptop, note what it was and how you fixed it. You will need it in Part 5.

## Part 4: A short serial test

Time your Problem 4 search on **Moby Dick only, strides 2 to 100**, once on your laptop and once on the Midway3 login node. Run it **serially**: no `multiprocessing` on the login node. The parallel version from Problem 5 runs on a compute node in Module 4.

Use the same code and the same command on both machines, for example by wrapping your search in `time.perf_counter()`.

| Machine | Processor | Time (s) |
|---|---|---|
| Laptop | | |
| Midway3 login node | | |

In [ ]:
from pathlib import Path
from collections import defaultdict
from time import perf_counter


def clean_text(text):
    return "".join(char.lower() for char in text if char.isalpha())


def load_text(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        return clean_text(file.read())


def load_words(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        return {
            word
            for line in file
            if len(word := clean_text(line.strip())) >= 4
        }


def build_word_index(all_words):
    word_index = defaultdict(list)

    for word in all_words:
        word_index[word[0]].append(word)

    return word_index


def find_words(els_text, word_index, names):
    for pos, char in enumerate(els_text):
        for word in word_index.get(char, []):
            if els_text.startswith(word, pos):
                yield pos, word, word in names


def find_phrases(entity_matches, window=25):
    phrases = []
    cluster = []

    for pos, word in entity_matches:
        if not cluster:
            cluster = [(pos, word)]

        elif pos - cluster[0][0] <= window:
            cluster.append((pos, word))

        else:
            if len(cluster) >= 2:
                phrases.append(cluster)

            cluster = [(pos, word)]

    if len(cluster) >= 2:
        phrases.append(cluster)

    return phrases


def process_book(text, book_name, word_index, names):
    output_dir = Path("../output/biblecode/benchmark")
    output_dir.mkdir(parents=True, exist_ok=True)

    words_path = output_dir / f"{book_name}-words.txt"
    phrases_path = output_dir / f"{book_name}-phrases.txt"

    with open(words_path, "w", encoding="utf-8") as words_file, \
         open(phrases_path, "w", encoding="utf-8") as phrases_file:

        # Strides 2 to 100 inclusive
        for stride in range(2, 101):
            els_text = text[::stride]
            entity_matches = []

            for pos, word, is_entity in find_words(
                els_text, word_index, names
            ):
                words_file.write(f"{stride},{pos},{word}\n")

                if is_entity:
                    entity_matches.append((pos, word))

            phrases = find_phrases(entity_matches, window=25)

            for phrase in phrases:
                values = [str(stride)]

                for pos, word in phrase:
                    values.extend([str(pos), word])

                phrases_file.write(",".join(values) + "\n")


# Load reference words
words = load_words("../data/words_10k.txt")
names = load_words("../data/names_and_dates.txt")

all_words = words | names
word_index = build_word_index(all_words)

# Load Moby Dick
text = load_text("../data/moby-dick.txt")

# Measure sequential execution time
start = perf_counter()

process_book(text, "moby-dick", word_index, names)

end = perf_counter()

print(f"Moby Dick (strides 2–100): {end - start:.4f} seconds")

Moby Dick (strides 2–100): 49.6260 seconds


| Machine | Processor | Time (s) |
|---|---|---|
| Laptop | Apple M1 Pro | 49.62|
| Midway3 login node | Intel Xeon Gold 6346 @ 3.10 GHz| 58.9853|

## Part 5: Reflection

1. Was the login node faster or slower than your laptop, and by how much?
2. Run `uptime` and `who | wc -l` on the login node. What is the load average, and how many people are signed in? How could that affect your timing?
3. Run your test a second time. Did the time change? Why is a login node a poor place to benchmark?
4. Did anything break when you moved your code to the RCC, and how did you fix it?

The login node was slower than my laptop by 9.3653 seconds. 

Load average was 13.55, and there were 56 people signed in. The load average indicates that the system is under heavy load, and the number of users could affect performance due to shared resources.

When i ran the test a second time, the time changed slightly (50.25), which shows that performance can vary on a login node due to other users' activities. A login node is a poor place to benchmark because it is shared and can have variable performance.

When I moved my code to the RCC, nothing broke, and I was able to run my code successfully after setting up the environment and dependencies correctly.

When I moved my code to the RCC, nothing broke, and I was able to run my code successfully after setting up the environment and dependencies correctly.